**Investments: Theory and Data Analysis**, Bates, Boyer, and Fletcher

# Chapter 6: A Short Introduction to Pandas

Pandas is the Python package we use to organize and analyze tabular data. Pandas stands for ``Panel Data'' which is an econometrics term for multi-dimensional data sets. Overall you can think of Pandas like a powerful Excel spreadsheet. This notebook introduces the parts of Pandas that you will need for the Chapter 6 Ken French data lab: creating a DataFrame, inspecting its rows and columns, selecting data, filtering rows, and calculating summary statistics. The examples use a small, made-up set of monthly portfolio returns so that the notebook can run without downloading data.

## Learning objectives

By the end of this notebook, you should be able to:

- Explain the difference between a Pandas `Series` and a `DataFrame`.
- Create a DataFrame and recognize its index, columns, and shape.
- Select one or more columns and filter rows with a condition.
- Create a new column from existing columns.
- Use Pandas methods such as `head`, `describe`, `mean`, and `std`.
- Recognize how these operations will be used with Ken French portfolio returns.

## What is Pandas?

A spreadsheet is a useful way to picture a Pandas **DataFrame**: it has rows, columns, and labels. Each column in a DataFrame is a **Series**, which is a one-dimensional collection of values with an index. In a financial data set, rows often represent dates and columns often represent portfolios, factors, or other variables.

The standard import gives Pandas the short name `pd`.

In [1]:
import pandas as pd

# A Series is a labeled, one-dimensional collection of values.
monthly_returns = pd.Series(
    [0.020, -0.015, 0.010, 0.005],
    index=['January', 'February', 'March', 'April'],
    name='Portfolio return'
)

monthly_returns

January     0.020
February   -0.015
March       0.010
April       0.005
Name: Portfolio return, dtype: float64

## Create a small portfolio-return DataFrame

The Ken French lab returns a DataFrame with dates in the index and portfolio returns in columns. We will create a similar table. Returns are written as decimals: `0.02` means a 2% monthly return.

In [2]:
# Create a date index for six made-up monthly observations.
dates = pd.date_range('2024-01-31', periods=6, freq='ME')

portfolio_returns = pd.DataFrame(
    {
        'Low-momentum': [0.020, -0.015, 0.010, 0.005, -0.010, 0.018],
        'High-momentum': [0.028, -0.005, 0.016, 0.012, -0.004, 0.025],
        'Market': [0.022, -0.010, 0.012, 0.008, -0.006, 0.020],
        'Risk-free': [0.004, 0.004, 0.004, 0.004, 0.004, 0.004],
    },
    index=dates
)

portfolio_returns

,Low-momentum,High-momentum,Market,Risk-free
2024-01-31,0.020,0.028,0.022,0.004
2024-02-29,-0.015,-0.005,-0.010,0.004
2024-03-31,0.010,0.016,0.012,0.004
2024-04-30,0.005,0.012,0.008,0.004
2024-05-31,-0.010,-0.004,-0.006,0.004
2024-06-30,0.018,0.025,0.020,0.004


## Inspect a DataFrame

A few attributes and methods are useful whenever you receive a new data set:

- `.shape` reports the number of rows and columns.
- `.columns` reports the column labels.
- `.index` reports the row labels.
- `.head()` displays the first rows.
- `.dtypes` reports the type of each column.

This is the same first inspection you can perform on the DataFrame returned by `fm.load_ken_french_data(...)`.

In [3]:
print('Shape:', portfolio_returns.shape)
print('Columns:', portfolio_returns.columns.tolist())
print('Index type:', type(portfolio_returns.index).__name__)
print('Column types:')
print(portfolio_returns.dtypes)

portfolio_returns.head(3)

Shape: (6, 4)
Columns: ['Low-momentum', 'High-momentum', 'Market', 'Risk-free']
Index type: DatetimeIndex
Column types:
Low-momentum     float64
High-momentum    float64
Market           float64
Risk-free        float64
dtype: object


,Low-momentum,High-momentum,Market,Risk-free
2024-01-31,0.020,0.028,0.022,0.004
2024-02-29,-0.015,-0.005,-0.010,0.004
2024-03-31,0.010,0.016,0.012,0.004


## Select columns and rows

Use square brackets to select data. A single column name returns a Series. A list of column names returns a smaller DataFrame. `.loc` selects by labels, while `.iloc` selects by integer position.

In [4]:
# Select one column: the result is a Series.
market_returns = portfolio_returns['Market']

# Select several columns: the result is a DataFrame.
two_portfolios = portfolio_returns[['Low-momentum', 'High-momentum']]

# Select rows from March through May by their date labels.
spring_returns = portfolio_returns.loc['2024-03-31':'2024-05-31']

print('One column:')
print(market_returns)
print('\nTwo columns:')
print(two_portfolios)
print('\nSelected rows:')
spring_returns

One column:
2024-01-31    0.022
2024-02-29   -0.010
2024-03-31    0.012
2024-04-30    0.008
2024-05-31   -0.006
2024-06-30    0.020
Freq: ME, Name: Market, dtype: float64

Two columns:
            Low-momentum  High-momentum
2024-01-31         0.020          0.028
2024-02-29        -0.015         -0.005
2024-03-31         0.010          0.016
2024-04-30         0.005          0.012
2024-05-31        -0.010         -0.004
2024-06-30         0.018          0.025

Selected rows:


,Low-momentum,High-momentum,Market,Risk-free
2024-03-31,0.010,0.016,0.012,0.004
2024-04-30,0.005,0.012,0.008,0.004
2024-05-31,-0.010,-0.004,-0.006,0.004


## Filter rows and create a new column

A condition creates a Boolean Series containing `True` and `False`. Putting that condition inside square brackets keeps only the rows where it is `True`. This is useful for questions such as “which months had a negative market return?”

A new column can be calculated from existing columns. The excess market return is the market return minus the risk-free return.

In [5]:
# Create a new column using element-by-element subtraction.
portfolio_returns['Market excess'] = (
    portfolio_returns['Market'] - portfolio_returns['Risk-free']
)

# Keep only months when the market return was negative.
negative_market_months = portfolio_returns[portfolio_returns['Market'] < 0]

negative_market_months[['Market', 'Risk-free', 'Market excess']]

,Market,Risk-free,Market excess
2024-02-29,-0.010,0.004,-0.014
2024-05-31,-0.006,0.004,-0.010


## Calculate summary statistics

Pandas methods apply calculations down each column by default. The mean and standard deviation below are monthly statistics. In the Ken French lab, you will annualize statistics when appropriate and compare portfolios across columns.

The `.describe()` method provides a compact collection of common statistics, including the count, mean, standard deviation, minimum, quartiles, and maximum.

In [6]:
summary = pd.DataFrame({
    'Mean': portfolio_returns[['Low-momentum', 'High-momentum']].mean(),
    'Monthly std. dev.': portfolio_returns[['Low-momentum', 'High-momentum']].std(),
})

print('Selected summary statistics:')
print(summary)
print('\nFull description:')
portfolio_returns[['Low-momentum', 'High-momentum']].describe()

Selected summary statistics:
                   Mean  Monthly std. dev.
Low-momentum   0.004667           0.014445
High-momentum  0.012000           0.014043

Full description:


,Low-momentum,High-momentum
count,6.000000,6.000000
mean,0.004667,0.012000
std,0.014445,0.014043
min,-0.015000,-0.005000
25%,-0.006250,0.000000
50%,0.007500,0.014000
75%,0.016000,0.022750
max,0.020000,0.028000


## A preview of the Ken French workflow

The Chapter 6 Ken French data lab will load historical data instead of the small table created above. The basic workflow is the same:

1. Load a DataFrame.
2. Inspect its index and columns.
3. Select portfolio columns.
4. Calculate statistics or new columns.
5. Pass the results to Matplotlib for visualization.

The exact column labels will depend on the strategy and portfolio sort you load.

## Key terms

- **Series:** a one-dimensional, labeled collection of values.
- **DataFrame:** a two-dimensional table with labeled rows and columns.
- **Index:** the labels for the rows of a Series or DataFrame.
- **Boolean filter:** a condition used to keep selected rows.
- **Method:** an action available on an object, such as `df.head()` or `df.mean()`.

## Next step

Continue with `Ex06-matplotlib_intro.ipynb` to learn how to visualize return data. Then open `Ex06-Ken French_Portfolio_Returns.ipynb` to apply Pandas and Matplotlib to historical portfolio returns from the Ken French Data Library.